# ⚔️ Apex-R 2.0 實戰監控中心（Google Colab 版）

與桌面版相同，可在雲端執行監控。**支援兩種策略一鍵切換：**

| 策略 ID | 說明 |
|---------|------|
| `apex_r` | 經典 Apex-R：120日動能 / Top12 |
| `apex_opt` | 2.0 優化：252日動能 / Top8 |

宇宙皆為 **S&P 500 兩年成交額 Top 65** + 手動台股。

---

## 使用步驟

1. **執行「安裝依賴」** cell
2. （可選）**掛載 Google Drive**，把 `colab/` 資料夾放在雲端硬碟
3. **調整下方表單**（日期、資金、標的、參數）
4. **執行「載入模組」** — 上傳 `colab/modules/` 內四個 `.py`，或 zip 整包 modules
5. **執行「執行監控」** 產生報告
6. **執行「下載報告」** 儲存 `.txt` 到本機

> 模組來自專案 `shared/`（本機執行 `sync_shared.ps1` 會同步到 `colab/modules/`），與桌面版、手機版共用同一份核心邏輯。

In [ ]:
# @title 📦 安裝依賴
!pip install yfinance pandas numpy pytz -q
print("✅ 依賴安裝完成")

In [ ]:
# @title 📂 （可選）掛載 Google Drive
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/apex-r/colab"  # @param {type:"string"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print(f"已掛載 Drive，Colab 路徑: {DRIVE_PROJECT_DIR}")
else:
    print("未使用 Drive，將於執行時上傳 modules/ 內的 .py 檔案")

In [ ]:
# @title 🎯 0. 策略版本
STRATEGY_ID = "apex_r"  # @param ["apex_r", "apex_opt"]
# apex_r = 經典 Apex-R (120d / Top12)  |  apex_opt = 2.0 優化 (252d / Top8)

# @title ⚙️ 1. 監控日期與資金（台幣）
QUERY_DATE = "2026-06-19"  # @param {type:"date"}
CURRENT_MONTH_CAPITAL_TWD = 1000000  # @param {type:"number"}
NEXT_MONTH_INJECTION_TWD = 0  # @param {type:"number"}
TOP_N_LIMIT = 12  # @param {type:"integer"}  # 切換策略後會自動覆寫
MOMENTUM_DAYS = 120  # @param {type:"integer"}
BUFFER_ZONE = 0.97  # @param {type:"number"}

In [ ]:
# @title 🎯 2. 投資宇宙（美股自動/手動 + 台股手動）
REFRESH_UNIVERSE = False  # @param {type:"boolean"}
UNIVERSE_SIZE = 65  # @param {type:"integer"}
UNIVERSE_MODE = "dollar_volume"  # @param ["dollar_volume", "momentum"]

USER_TICKERS = """NVDA, TSLA, AAPL, MSFT, META, AMZN, AMD, PLTR, AVGO, GOOGL,
SMCI, NFLX, UNH, LLY, MU, TSM, ORCL, JPM, INTC, V,
BABA, CRM, COST, HOOD, XOM, BAC, ADBE, BA, WMT, UBER,
CRWD, QCOM, MA, HD, AMAT, JNJ, GS, NOW, ASML, CVX,
MRVL, PG, CSCO, MRK, PANW, TXN, WFC, PDD, PFE, ABBV,
GE, LRCX, DIS, C, NKE, PEP, INTU, IBM, KO, ARM,
CAT, MCD, SBUX, LIN, SHOP"""  # @param {type:"string"}

TW_TICKERS = "0050.TW, 2330.TW, 2317.TW, 2454.TW"  # @param {type:"string"}

In [ ]:
# @title ⚖️ 3. 其他策略參數（通常無需改）
MAX_WEIGHT_CAP = 0.10  # @param {type:"number"}
VOLATILITY_DAYS = 20  # @param {type:"integer"}
MA_DAYS = 200  # @param {type:"integer"}
TRANSACTION_COST = 0.002  # @param {type:"number"}
BREADTH_THRESHOLD = 0.50  # @param {type:"number"}
SHOW_MARKET_MODE = True  # @param {type:"boolean"}

In [ ]:
# @title 🔧 載入 Apex-R 模組
import io
import os
import shutil
import sys
import zipfile

MODULES_DIR = "modules"
REQUIRED_MODULES = ["apex_core.py", "universe_builder.py", "strategy_presets.py", "config_io.py"]

for p in [".", MODULES_DIR, "/content", "/content/modules"]:
    if p not in sys.path:
        sys.path.insert(0, p)


def _module_path(name: str) -> str:
    in_modules = os.path.join(MODULES_DIR, name)
    if os.path.isfile(in_modules):
        return in_modules
    return name


def _missing_modules():
    return [f for f in REQUIRED_MODULES if not os.path.isfile(_module_path(f))]


def _copy_from_drive():
    if not globals().get("USE_DRIVE"):
        return
    drive_dir = globals().get("DRIVE_PROJECT_DIR", "")
    modules_src = os.path.join(drive_dir, "modules")
    if not os.path.isdir(modules_src):
        modules_src = drive_dir
    if not drive_dir or not os.path.isdir(drive_dir):
        print(f"⚠️ Drive 路徑不存在: {drive_dir}")
        return
    os.makedirs(MODULES_DIR, exist_ok=True)
    for name in REQUIRED_MODULES:
        src = os.path.join(modules_src, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(MODULES_DIR, name))
            print(f"  已複製 {name}")


def _upload_modules():
    from google.colab import files

    print("請上傳 colab/modules/ 內四個 .py，或 modules.zip：")
    uploaded = files.upload()
    os.makedirs(MODULES_DIR, exist_ok=True)
    for name, data in uploaded.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                zf.extractall(MODULES_DIR)
        else:
            dest = os.path.join(MODULES_DIR, name) if name.endswith(".py") else name
            with open(dest, "wb") as f:
                f.write(data)


missing = _missing_modules()
if missing:
    _copy_from_drive()
    missing = _missing_modules()

if missing:
    _upload_modules()
    missing = _missing_modules()

if missing:
    raise FileNotFoundError(f"仍缺少模組: {missing}")

from apex_core import ApexConfig, format_report, run_monitor
from strategy_presets import apply_preset, report_header
from universe_builder import build_universe, parse_ticker_csv

print("✅ 模組載入完成")

In [ ]:
# @title ▶️ 執行監控
us_tickers = USER_TICKERS.replace("\n", " ").strip()
tw_tickers = TW_TICKERS.replace("\n", " ").strip()
universe_last_updated = ""

if REFRESH_UNIVERSE:
    print(f"🔄 更新美股宇宙 (Top {UNIVERSE_SIZE}, 模式 {UNIVERSE_MODE})… 約需 1～3 分鐘\n")
    tw_list = parse_ticker_csv(tw_tickers)
    uresult = build_universe(
        as_of=str(QUERY_DATE),
        top_n=int(UNIVERSE_SIZE),
        tw_tickers=tw_list,
        mode=UNIVERSE_MODE,
        momentum_days=504,
    )
    us_tickers = ", ".join(uresult.us_tickers)
    universe_last_updated = uresult.as_of
    for line in uresult.summary_lines():
        print(line)
    print()

cfg = ApexConfig(
    query_date=str(QUERY_DATE),
    current_month_capital_twd=float(CURRENT_MONTH_CAPITAL_TWD),
    next_month_injection_twd=float(NEXT_MONTH_INJECTION_TWD),
    user_tickers=us_tickers,
    tw_tickers=tw_tickers,
    strategy_id=str(STRATEGY_ID),
    universe_size=int(UNIVERSE_SIZE),
    universe_mode=UNIVERSE_MODE,
    universe_momentum_days=504,
    universe_last_updated=universe_last_updated,
    transaction_cost=float(TRANSACTION_COST),
    max_weight_cap=float(MAX_WEIGHT_CAP),
    top_n_limit=int(TOP_N_LIMIT),
    buffer_zone=float(BUFFER_ZONE),
    volatility_days=int(VOLATILITY_DAYS),
    momentum_days=int(MOMENTUM_DAYS),
    ma_days=int(MA_DAYS),
    breadth_threshold=float(BREADTH_THRESHOLD),
    show_market_mode=bool(SHOW_MARKET_MODE),
)
cfg = apply_preset(cfg, STRATEGY_ID)

print(f"📋 {report_header(cfg.strategy_id)}")
print(f"📥 宇宙 {len(cfg.ticker_list())} 檔 | Top{cfg.top_n_limit} | {cfg.momentum_days}d 動能 | 查詢日 {cfg.query_date}\n")
result = run_monitor(cfg)
REPORT_TEXT = format_report(result, strategy_id=cfg.strategy_id)
print(REPORT_TEXT)

In [ ]:
# @title 💾 下載報告 (.txt)
from google.colab import files

if "REPORT_TEXT" not in globals() or not REPORT_TEXT:
    raise RuntimeError("請先執行「執行監控」cell")

out_name = f"apex_r_report_{QUERY_DATE}.txt"
with open(out_name, "w", encoding="utf-8") as f:
    f.write(REPORT_TEXT)

files.download(out_name)
print(f"✅ 已下載 {out_name}")

In [ ]:
# @title 📋 （可選）複製更新後的美股清單
if "us_tickers" in globals():
    print("# 貼回設定用的美股清單：\n")
    print(us_tickers)
else:
    print("尚未執行監控；若勾選了「更新宇宙」，請先執行監控 cell。")